<a href="https://colab.research.google.com/github/yueop/AS_LAB/blob/main/datasets_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')   #구글 드라이브 연결
Project_Path = '/content/drive/MyDrive/AS_LAB'  #저장할 폴더 경로 설정

#저장할 폴더 존재 여부 확인(없으면 새 폴더 생성, 있으면 기존 폴더 사용)
if not os.path.exists(Project_Path):
    os.makedirs(Project_Path)
    print(f"새 폴더 생성 후 저장 완료: {Project_Path}")
else:
    print(f"기존 폴더에 저장: {Project_Path}")

os.chdir(Project_Path)   #작업 위치 이동(change directory)

Mounted at /content/drive
기존 폴더에 저장: /content/drive/MyDrive/AS_LAB


In [ ]:
%%writefile datasets.py
import torch
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler #IRIS 정규화 용도

class UnifiedDataLoader:
    def __init__(self, batch_size=64):
        self.batch_size = batch_size
        self.transform = transforms.Compose([transforms.ToTensor(), #이미지 데이터 텐서로 변환(전처리)
                                             transforms.Normalize((0.5,), (0.5,))]) #데이터 정규화(-1.0~1.0 사이의 값으로 변환)

    def get_loader(self, dataset_name, is_train=True):
        dataset_name = dataset_name.lower()
        if dataset_name == 'iris':
            return self.get_iris_loader(is_train)
        elif dataset_name == 'mnist':
            return self.get_mnist_loader(is_train)
        elif dataset_name == 'fashionmnist':
            return self.get_fashionmnist_loader(is_train)
        else:
            print(f'지원하지 않는 데이터셋입니다.: {dataset_name}') #예외 처리
            return None

    def get_iris_loader(self, is_train): #IRIS 데이터셋 로더
        iris = load_iris() #사이킷런 라이브러리에서 데이터 로드
        X, y = iris.data, iris.target

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42) #데이터 분할

        scaler = StandardScaler() #IRIS 데이터 정규화
        X_train = scalar.fit_transform(X_train)
        X_test = scalar.transform(X_test)

        if is_train: #학습용 데이터와 테스트 데이터 구분
            X_tensor = torch.tensor(X_train, dtype=torch.float32)
            y_tensor = torch.tensor(y_train, dtype=torch.long)
        else:
            X_tensor = torch.tensor(X_test, dtype=torch.float32)
            y_tensor = torch.tensor(y_test, dtype=torch.long)

        dataset = TensorDataset(X_tensor, y_tensor)
        return DataLoader(dataset, batch_size = self.batch_size, shuffle=is_train)

    def get_mnist_loader(self, is_train): #MNIST 데이터셋 로더
        dataset = datasets.MNIST(root='./data',
                                 train=is_train,
                                 download=True,
                                 transform=self.transform)
        return DataLoader(dataset, batch_size = self.batch_size, shuffle=is_train)

    def get_fashionmnist_loader(self, is_train): #FashionMNIST 데이터셋 로더(MNIST와 같은 구조)
        dataset = datasets.FashionMNIST(root='.data',
                                        train=is_train,
                                        download=True,
                                        transform=self.transform)
        return DataLoader(dataset, batch_size=self.batch_size, shuffle=is_train)

Writing datasets.py
